This notebook is to use a cross-encoder to rerank retrievals for a query.

Collect 100 retrievals (using any retrieval, primarily hybrid) for each question. The cross-encoder reranks these retrievals and chooses the top 10 most relevant ones.

upload corpus.jsonl and retrievals/hybrid_search_retrievals_k100.csv


In [1]:
!pip install -q transformers
from google.colab import files

In [2]:
!nvidia-smi
import torch; print(torch.cuda.is_available())

Sat Nov  7 15:19:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import ast, pandas as pd
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEVICE = "cuda"
tok = AutoTokenizer.from_pretrained("ncbi/MedCPT-Cross-Encoder")
model = AutoModelForSequenceClassification.from_pretrained("ncbi/MedCPT-Cross-Encoder").to(DEVICE).half().eval()

CORPUS = pd.read_json("corpus.jsonl", lines=True, dtype={"pmid": str}).set_index("pmid")
qs = pd.read_csv("hybrid_search_retrievals_k100.csv")
qs["retrievals"] = qs["retrievals"].apply(ast.literal_eval)   # "[1, 2, ...]" text -> real list
print(len(qs), "questions,", len(qs["retrievals"][0]), "candidates each")

Rerank:

In [ ]:
def rerank(question, pmids, k=10, chunk=50, max_length=512):
    docs = CORPUS.loc[[str(p) for p in pmids]]
    texts = [f"{t} {a}" for t, a in zip(docs["title"], docs["abstract"])]
    scores = []
    with torch.no_grad():
        for i in range(0, len(texts), chunk):
            pairs = [[question, t] for t in texts[i:i + chunk]]
            enc = tok(pairs, truncation=True, padding=True, max_length=max_length, return_tensors="pt").to(DEVICE)
            scores.extend(model(**enc).logits.squeeze(dim=1).float().tolist())
    ranked = sorted(zip(pmids, scores), key=lambda x: -x[1])
    return [p for p, _ in ranked[:k]]


In [ ]:
# One pass per max_length setting -> three comparable reranked_top10_ml*.csv files.
# 512 is the original default (longer abstracts aren't truncated, but every chunk pads
# up to whichever document in it is longest -> slower). 256/128 trade quality for speed.
for max_length in (512, 256, 128):
    qs[f"reranked"] = [
        rerank(q, r, max_length=max_length)
        for q, r in tqdm(zip(qs["question"], qs["retrievals"]), total=len(qs), desc=f"max_length={max_length}")
    ]
    out = f"reranked_top10_ml{max_length}.csv"
    qs[["question", "qid", "type", "reranked"]].to_csv(out, index=False)
    print("saved", out)
    # files.download(out)
